In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers.modeling_outputs import BaseModelOutput

from transformers import (
    CLIPVisionModel,
    CLIPVisionConfig,
    ViTModel,
    T5EncoderModel,
    T5ForConditionalGeneration,
    T5TokenizerFast,
    CLIPProcessor
)


from peft import LoraConfig, get_peft_model, TaskType
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os 


In [2]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TEST_IMAGE_DIR,
                            FAISS_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CLIP_MODEL_NAME,
                            T5_MODEL_NAME,
                            CHECKPOINT_DIR)

# TRAIN_IMAGE_DIR = "dataset/flickr30k_images/train"
# TRAIN_CAPTIONS_DIR = "dataset/captions-train.csv"

# TEST_IMAGE_DIR = "dataset/flickr30k_images/test"
# TEST_CAPTIONS_DIR = "dataset/captions-test.csv"

# FAISS_PATH = "flickr30k_clip_images.faiss"
# TRAIN_METADATA_PATH = "train_metadata.json"
# TEST_METADATA_PATH = "test_metadata.json"

# CHECKPOINT_DIR = "FusionVLM"

from Modules.FusionVLM import FusionVLM
from Modules.retrieval_module import Retriever
from Modules.datasets import VLMDataset, VLMDataCollator
from Modules.utils import print_model_param_stats

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
CLIP_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, use_fast=True)
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME)
collator = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)

In [5]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [6]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [7]:
model = FusionVLM(
    vision_encoder_name="openai/clip-vit-base-patch32",
    text_encoder_name="t5-base",
    text_decoder_name="t5-base",
    num_fusion_blocks=4,
    use_local_files=True
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

Total parameters: 458,385,792


In [8]:
def freeze_module(module: torch.nn.Module):
    for p in module.parameters():
        p.requires_grad = False

# Freeze vision encoder
freeze_module(model.vision_encoder)

# Freeze text encoder
freeze_module(model.text_encoder)

In [9]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        # Decoder self-attention (language modeling)
        "SelfAttention.q",
        "SelfAttention.v",

        # Decoder cross-attention (fusion output → text)
        "EncDecAttention.q",
        "EncDecAttention.v",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model.text_decoder = get_peft_model(model.text_decoder, lora_config)
model.text_decoder.print_trainable_parameters()


trainable params: 3,538,944 || all params: 226,442,496 || trainable%: 1.5628


In [10]:
# Unfreeze the lm head for token generation
for param in model.text_decoder.lm_head.parameters():
    print(param.shape[0]*param.shape[1])
    param.requires_grad = True

24674304


In [11]:
print_model_param_stats(model)

Module                                          Total    Trainable       Frozen
--------------------------------------------------------------------------------
vision_encoder                             87,456,000            0   87,456,000
text_encoder                              109,628,544            0  109,628,544
text_decoder                              226,442,496   28,213,248  198,229,248
vision_proj                                   590,592      590,592            0
fusion_blocks                              37,807,104   37,807,104            0
--------------------------------------------------------------------------------
TOTAL                                     461,924,736   66,610,944  395,313,792


In [12]:
NUM_EPOCHS = 1
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

scaler = GradScaler()

In [16]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model.train()
loss_history = []

for epoch in range(NUM_EPOCHS):
    batch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)

    for batch in progress_bar:
        optimizer.zero_grad()

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(
                query_pixel_values=batch["query_pixel_values"],
                retrieved_pixel_values=batch["retrieved_pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    epoch_loss = batch_loss / len(train_loader)
    loss_history.append(epoch_loss)
    
    progress_bar.set_postfix(final_loss=epoch_loss)
    print(f"Epoch {epoch+1} Avg Loss: {epoch_loss:.4f}")

    # ---- Backup every 5 epochs ----
    if (epoch + 1) % 5 == 0:
        backup_path = os.path.join(CHECKPOINT_DIR, f"FusionVLM_epoch{epoch+1}.pt")
        torch.save(model.state_dict(), backup_path)


Epoch 1:   2%|▏         | 36/1924 [00:19<16:43,  1.88it/s, loss=3.72]


KeyboardInterrupt: 

In [15]:
model.eval()

for batch in train_loader:
    with torch.no_grad():
        outputs = model(
                        query_pixel_values=batch["query_pixel_values"],
                        retrieved_pixel_values=batch["retrieved_pixel_values"],
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                        labels=batch["labels"]
                        # labels=None
                    )
    break

logits = outputs.logits

pred_ids = torch.argmax(logits, dim=-1)

texts = T5_tokenizer.batch_decode(
    pred_ids,
    skip_special_tokens=True
)

print(texts)

[" -' a - and  to a  with  with a -  ..               ", ")- or' a   jacket is    for for be  out. the air..          ", '                                       ', "ufgrund man' a   can   orange print on is is face face is- she     ", 'ufgrundster who a disability character is the or is  a y in. -s. a bar his back ofpaw       ', "f'whose claim  a  to a  onaning to her own  lead to-                 ", 'ufgrund man in a  carpet is   sister in  to thea  populateded path to.        ', 'f andlooking,-icecot is a   on  a - .  the who thea time event in are are -s of their faces.- ', 'ufgrundfield whiteaan    around the water.-         ', '                                       ', 'ufgrund but in a t is  out thet weathersands thea beach in.             ', 'ufgrundsterentrepreneurke man who his completingecuringrubbing his of inghes             ', 'f who a hurry jackethatandeve is  can  in to  important game and-          ', '                                       ', 'im  and is  to  first as 

In [ ]:
outputs.logits.shape

In [ ]:
model.eval()

for batch in test_loader:
    with torch.no_grad():
        generated_ids = model.generate(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch["retrieved_pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=4
        )
        
        captions = T5_tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        print(captions)
        
    break


e e                                                                                                                                                                                                                                                                                tul                                                                                                      tul tultul tultul tultul tultultul  tultultul tultultul tultultul tultultul tultultul tultultul tultultul tultul tultul tultul tultul tultul tultul    e                                                                                                                                                                                                                                                                                                                        tultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultultul

In [ ]:
generated_ids